# No.6 画像デノイジング — BM3D

BM3D（Block-Matching and 3D collaborative filtering）でMRI画像のノイズを除去します。

**処理の流れ:**
1. ノイズ付加 (`add_noise.py` で生成)
2. BM3D によるデノイズ
3. PSNR / SSIM で品質評価

In [ ]:
import numpy as np
import bm3d
import skimage.io
import skimage.metrics
import matplotlib.pyplot as plt
import japanize_matplotlib


## ノイズ付加

In [ ]:
noise_sigma = 0.05
A = skimage.io.imread('data/original.pgm').astype(float) / 255.0

rng = np.random.default_rng(seed=0)
noise = rng.normal(0, noise_sigma, A.shape)
An = np.clip(A + noise, 0, 1)

psnr_noisy = skimage.metrics.peak_signal_noise_ratio(A, An, data_range=1.0)
print(f'Noisy PSNR: {psnr_noisy:.2f} dB')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(A, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(An, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Noisy (σ={noise_sigma}, PSNR={psnr_noisy:.1f}dB)')
axes[1].axis('off')
plt.show()

## BM3D デノイジング

**MATLABとの対応:**
```matlab
[~, denoised] = BM3D(1, single(An), 255*noiseSigma/sqrt(2));
```
```python
denoised = bm3d.bm3d(An, sigma_psd=noise_sigma/np.sqrt(2))
```

In [ ]:
# sigma_psd: [0,1]スケールで指定。MATLABのsqrt(2)補正と同じ
sigma_psd = noise_sigma / np.sqrt(2)
denoised = bm3d.bm3d(An, sigma_psd=sigma_psd)

psnr = skimage.metrics.peak_signal_noise_ratio(A, denoised, data_range=1.0)
ssim = skimage.metrics.structural_similarity(A, denoised, data_range=1.0)
print(f'BM3D PSNR: {psnr:.2f} dB')
print(f'BM3D SSIM: {ssim:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, im, title in zip(axes, [A, An, denoised],
                          ['Original', f'Noisy ({psnr_noisy:.1f}dB)',
                           f'BM3D ({psnr:.1f}dB, SSIM={ssim:.3f})']):
    ax.imshow(np.clip(im, 0, 1), cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')
fig.tight_layout()
plt.show()